# End-to-End Workflow

This notebook runs the workflow as a model matrix instead of a single chain.

- Temporal demand `D(t)` variants: Random Forest and XGBoost
- POI attractiveness `A(p)` variants: manual baseline, Random Forest, and XGBoost
- Integration rule: `CrowdIndex(p, t) = D(t) * A(p)`


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'processed'
model_df = pd.read_csv(DATA / 'model_dataset_with_holidays.csv', parse_dates=['date'])
manual = pd.read_csv(DATA / 'otm_poi_weights_manual_final.csv')
rf_partb = pd.read_csv(DATA / 'otm_part_b_rf_scores.csv')
xgb_partb = pd.read_csv(DATA / 'otm_part_b_xgb_scores.csv')


In [2]:
X = model_df.drop(columns=['date', 'crowd_index', 'crowd_level'])
y = model_df['crowd_index']
split_index = int(len(model_df) * 0.8)
X_train = X[:split_index]
y_train = y[:split_index]

rf_model = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
rf_model.fit(X_train, y_train)
pred_rf = model_df.copy()
pred_rf['predicted_crowd'] = rf_model.predict(X)
pred_rf.to_csv(DATA / 'predictions_rf.csv', index=False)

xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6, random_state=42)
xgb_model.fit(X_train, y_train)
pred_xgb = model_df.copy()
pred_xgb['predicted_crowd'] = xgb_model.predict(X)
pred_xgb.to_csv(DATA / 'predictions_xgb.csv', index=False)
pred_xgb.to_csv(DATA / 'predictions.csv', index=False)


In [3]:
poi_cols = [
    'poi_id', 'display_name_en', 'category_clean', 'query_area', 'lat', 'lon',
    'has_direct_wiki_signal', 'exclude_from_part_b', 'is_mosque', 'is_church',
    'is_synagogue', 'is_cathedral', 'is_palace', 'is_museum', 'is_monument',
    'is_cemetery', 'is_fortress', 'is_tower', 'is_hamam', 'is_bridge',
    'is_tekke_or_dergah', 'is_tomb', 'is_fountain', 'is_gate', 'subtype_flag_count'
]

manual_pois = manual[poi_cols + ['poi_weight', 'poi_weight_source', 'poi_weight_confidence']].copy()
manual_pois = manual_pois.rename(columns={'poi_weight': 'poi_weight_value', 'poi_weight_source': 'score_source', 'poi_weight_confidence': 'score_confidence'})
manual_pois['part_b_model'] = 'manual'

rf_pois = rf_partb[poi_cols + ['A_rf', 'score_source_rf', 'observed_proxy_score', 'rf_predicted_proxy_score', 'final_score_rf']].copy()
rf_pois = rf_pois.rename(columns={'A_rf': 'poi_weight_value', 'score_source_rf': 'score_source', 'rf_predicted_proxy_score': 'predicted_proxy_score', 'final_score_rf': 'final_proxy_score'})
rf_pois['part_b_model'] = 'rf'
rf_pois['score_confidence'] = np.where(rf_pois['score_source'].eq('observed_wiki'), 'high', 'model_imputed')

xgb_pois = xgb_partb[poi_cols + ['A_xgb', 'score_source_xgb', 'observed_proxy_score', 'xgb_predicted_proxy_score', 'final_score_xgb']].copy()
xgb_pois = xgb_pois.rename(columns={'A_xgb': 'poi_weight_value', 'score_source_xgb': 'score_source', 'xgb_predicted_proxy_score': 'predicted_proxy_score', 'final_score_xgb': 'final_proxy_score'})
xgb_pois['part_b_model'] = 'xgb'
xgb_pois['score_confidence'] = np.where(xgb_pois['score_source'].eq('observed_wiki'), 'high', 'model_imputed')


In [4]:
def prepare_temporal(df, label):
    temp = df[['date', 'predicted_crowd', 'crowd_index', 'crowd_level', 'trend_demand', 'temp_max', 'temp_min', 'temp_avg', 'precipitation', 'month', 'week_of_year', 'season_spring', 'season_summer', 'season_winter', 'is_holiday']].copy()
    temp = temp.rename(columns={'predicted_crowd': 'city_demand_score', 'crowd_index': 'observed_city_crowd_index', 'crowd_level': 'city_crowd_level'})
    temp['temporal_model'] = label
    return temp

rf_temporal = prepare_temporal(pred_rf, 'rf')
xgb_temporal = prepare_temporal(pred_xgb, 'xgb')

def build_combo(temporal_df, poi_df, temporal_label, partb_label):
    rows = []
    for _, t in temporal_df.iterrows():
        tmp = poi_df.copy()
        for c in temporal_df.columns:
            tmp[c] = t[c]
        tmp['crowdindex_poi'] = tmp['city_demand_score'] * tmp['poi_weight_value']
        tmp['temporal_model'] = temporal_label
        tmp['part_b_model'] = partb_label
        rows.append(tmp)
    out = pd.concat(rows, ignore_index=True)
    out['workflow_model'] = out['temporal_model'] + '__' + out['part_b_model']
    out['poi_crowd_rank_desc'] = out.groupby(['date', 'workflow_model'])['crowdindex_poi'].rank(method='first', ascending=False)
    out['poi_crowd_percentile_desc'] = out.groupby(['date', 'workflow_model'])['crowdindex_poi'].rank(pct=True, ascending=False)
    out['poi_busyness_band'] = pd.cut(out['poi_crowd_percentile_desc'], bins=[0, 0.2, 0.5, 0.8, 1.0], labels=['Very High', 'High', 'Medium', 'Low'], include_lowest=True)
    return out


In [5]:
part_b_tables = {'manual': manual_pois, 'rf': rf_pois, 'xgb': xgb_pois}
combo_frames = {}
for temporal_label, temporal_df in [('rf', rf_temporal), ('xgb', xgb_temporal)]:
    for partb_label, poi_df in part_b_tables.items():
        combo_name = f'{temporal_label}__{partb_label}'
        combo_frames[combo_name] = build_combo(temporal_df, poi_df, temporal_label, partb_label)
        combo_frames[combo_name].to_csv(DATA / f'otm_crowdindex_{combo_name}_weekly.csv', index=False)
combined = pd.concat(combo_frames.values(), ignore_index=True)
combined.to_csv(DATA / 'otm_crowdindex_workflow_matrix_weekly.csv', index=False)
combined[['workflow_model']].value_counts()


In [6]:
summary = {
    'working_pois': int(manual.shape[0]),
    'weeks': int(model_df.shape[0]),
    'workflow_variants': sorted(combo_frames.keys()),
    'rows_per_variant': int(next(iter(combo_frames.values())).shape[0]),
    'combined_rows': int(combined.shape[0]),
}
summary


## Recommendation Example

This example uses `xgb__rf`: XGBoost for temporal demand and Random Forest for Part B attractiveness.


In [7]:
example_df = combo_frames['xgb__rf']
busiest_date = xgb_temporal.sort_values('city_demand_score', ascending=False).iloc[0]['date']
anchor_name = 'Hagia Sophia'
day_df = example_df[example_df['date'] == busiest_date].copy()
anchor = day_df[day_df['display_name_en'] == anchor_name].iloc[0]
subtype_cols = [
    'is_mosque', 'is_church', 'is_synagogue', 'is_cathedral', 'is_palace', 'is_museum',
    'is_monument', 'is_cemetery', 'is_fortress', 'is_tower', 'is_hamam', 'is_bridge',
    'is_tekke_or_dergah', 'is_tomb', 'is_fountain', 'is_gate'
]
def haversine(lat1, lon1, lat2, lon2):
    r = 6371.0
    p1 = np.radians(lat1)
    p2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)
    a = np.sin(dphi / 2.0) ** 2 + np.cos(p1) * np.cos(p2) * np.sin(dl / 2.0) ** 2
    return 2 * r * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
candidates = day_df[(day_df['category_clean'] == anchor['category_clean']) & (day_df['poi_id'] != anchor['poi_id'])].copy()
candidates['subtype_overlap_count'] = candidates[subtype_cols].mul(anchor[subtype_cols].values, axis=1).sum(axis=1)
candidates['distance_to_anchor_km'] = haversine(anchor['lat'], anchor['lon'], candidates['lat'].values, candidates['lon'].values)
candidates['is_quieter_than_anchor'] = candidates['crowdindex_poi'] < anchor['crowdindex_poi']
quieter = candidates[candidates['is_quieter_than_anchor']].copy()
crowd_min = quieter['crowdindex_poi'].min()
crowd_max = quieter['crowdindex_poi'].max()
dist_min = quieter['distance_to_anchor_km'].min()
dist_max = quieter['distance_to_anchor_km'].max()
overlap_max = max(int(quieter['subtype_overlap_count'].max()), 1)
quieter['crowd_relief_score'] = 1 - ((quieter['crowdindex_poi'] - crowd_min) / (crowd_max - crowd_min + 1e-9))
quieter['distance_score'] = 1 - ((quieter['distance_to_anchor_km'] - dist_min) / (dist_max - dist_min + 1e-9))
quieter['subtype_similarity_score'] = quieter['subtype_overlap_count'] / overlap_max
quieter['recommendation_score'] = 0.45 * quieter['crowd_relief_score'] + 0.35 * quieter['subtype_similarity_score'] + 0.20 * quieter['distance_score']
example = quieter.sort_values(['recommendation_score', 'crowd_relief_score'], ascending=False).head(15).copy()
example['anchor_poi'] = anchor_name
example['anchor_date'] = pd.Timestamp(busiest_date).strftime('%Y-%m-%d')
example['workflow_model'] = 'xgb__rf'
example.to_csv(DATA / 'otm_recommendation_example_xgb_rf.csv', index=False)
example[['display_name_en', 'query_area', 'crowdindex_poi', 'distance_to_anchor_km', 'subtype_overlap_count', 'recommendation_score']].head(10)
